In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb

# ============================================================
# data loading
# ============================================================

snapshot_id = "20260518_011926_d4b20816"
DATA_PATH = f"./data/raw/{snapshot_id}.parquet"
df = pd.read_parquet(DATA_PATH)

# ============================================================
# data cleaning
# ============================================================

# type conversion
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])
df["duration_minutes"] = pd.to_numeric(df["duration"])/60
df["Power_kW"] = pd.to_numeric(df["PricePerKWh"])
df["consumption_kwh"] = pd.to_numeric(df["consumption_kwh"])
df["connector_id"] = df["connector_id"].astype(int)
df["latitude"] = pd.to_numeric(df["latitude"])
df["longitude"] = pd.to_numeric(df["longitude"])


# Remove short charging session and low consumption
df = df[
    (df["consumption_kwh"] > 0)
    &
    (df["duration_minutes"] > 1)
]

# Drop missing coordinates/city as we are classifying based on city
df = df.dropna(
    subset=[
        "City",
        "latitude",
        "longitude"
    ]
)

# ============================================================
# feature engineering
# ============================================================

df["hour"] = df["start_time"].dt.hour
df["dayofweek"] = df["start_time"].dt.dayofweek
df["month"] = df["start_time"].dt.month
df["is_weekend"] = (
    df["dayofweek"] >= 5
).astype(int)


# Used city for clustering
df["cluster"] = df["City"]

# Use connector_id approximates actual charging slots.
connectors_per_cluster = (
    df.groupby("cluster")["connector_id"]
      .nunique()
      .reset_index(name="num_connectors")
)
connectors_per_cluster

In [ ]:
from ydata_profiling import ProfileReport
ProfileReport(df)

Based on the analysis above, there seems to be way too many duplicates in the data. After revising the harvester that I built, the duplication might come from the source iceberg from SENSE.

In [ ]:
df[df.duplicated(keep=False) & (df['cluster'] == 'Lanark') & (df['duration'] == 4755.0)]

Comparison of the two busiest cities

In [ ]:
glasgow = df[df['cluster'] == 'Glasgow']
dundee = df[df['cluster'] == 'Dundee']

glasgow_profile = ProfileReport(glasgow)
dundee_profile = ProfileReport(dundee)

In [ ]:
glasgow_profile.compare(dundee_profile)